# Flujo para Anfitriones

> Host: "Que puedo mejorar en mis listings para atraer mas huespedes"

* Revisa amenities del listing
* Compara contra competidores usando modelos de competicion
* Analiza reviews negativas usando llm
* Evalua calidad de la imagen con model CLIP
* Usa RAG para buscar patrones

In [92]:
import pandas as pd
import numpy as np
import sqlite3
import joblib
import warnings
import seaborn as sns

In [93]:
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_style("ticks")
odx = pd.IndexSlice
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

## Load host amenities from database

In [ ]:
# Load listings from database
def load_listings(db_path):
    conn = sqlite3.connect(db_path)
    query = "SELECT * FROM listings"
    listings_df = pd.read_sql_query(query, conn)
    conn.close()
    return listings_df

listings_df = load_listings('db/airbnb.db')

In [95]:
# get amenities from listings
listings_df['amenities_parsed']

0        Kitchen, Resort access, Hot water, Courtyard v...
1        Free street parking, Free parking on premises,...
2        Dining table, Hot water, Hangers, Essentials, ...
3        Hot water, TV with standard cable, Hangers, Es...
4        Varies conditioner, Dining table, Free street ...
                               ...                        
23122    Self check-in, Carbon monoxide alarm, Washer, ...
23123    Air conditioning, Kitchen, Smoke alarm, Exteri...
23124    Carbon monoxide alarm, First aid kit, Kitchen,...
23125    Dining table, Free street parking, Pool table,...
23126    Dining table, Free parking on premises, Condit...
Name: amenities_parsed, Length: 23127, dtype: object

## Amenities comparision

In [96]:
import os
from collections import Counter
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import joblib

# Construye un modelo de competidores similares y compara amenidades
# Usa listings_df ya cargado en el notebook.


def extract_amenities_sets(df):
    # convierte la columna amenities_parsed en sets
    return df['amenities_parsed'].fillna('').apply(lambda s: {a.strip().lower() for a in s.split(',') if a.strip()})

def build_amenity_features(df, top_n=40):
    amen_sets = extract_amenities_sets(df)
    all_amen = Counter()
    amen_sets.apply(all_amen.update)
    top_amen = [a for a, _ in all_amen.most_common(top_n)]
    # crear columnas booleanas para cada amenidad top
    for a in top_amen:
        col = f'amenity__{a}'
        df[col] = amen_sets.apply(lambda s, a=a: int(a in s))
    return df, top_amen

def build_feature_matrix(df, amen_list):
    # columnas numéricas básicas
    num_cols = ['latitude', 'longitude', 'price', 'amenities_n',
                'review_scores_rating', 'accommodates', 'beds', 'bedrooms', 'bathrooms']
    # asegurar existencia y rellenar NA
    X_num = df[num_cols].fillna(0).astype(float)
    X_amen = df[[f'amenity__{a}' for a in amen_list]].astype(float)
    X = pd.concat([X_num, X_amen], axis=1)
    return X, num_cols

def train_or_load_knn(X, model_path='knn_competitors.joblib', n_neighbors=50):
    # escala y entrena NearestNeighbors (o carga si existe)
    if os.path.exists(model_path):
        obj = joblib.load(model_path)
        return None, obj['knn']
    # scaler = StandardScaler()
    # Xs = scaler.fit_transform(X)
    knn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean')
    knn.fit(X)
    joblib.dump({'knn': knn}, model_path)
    return None, knn

def get_competitors(listing_id, df, X, scaler, knn, amen_list, k=10):
    # retorna DataFrame con competidores y comparativa de amenidades
    if listing_id not in df['id'].values:
        raise ValueError(f"listing_id {listing_id} no encontrado")
    idx = int(df.index[df['id'] == listing_id][0])
    Xs = X # scaler.transform(X)
    distances, indices = knn.kneighbors([X.iloc[idx]], n_neighbors=k+1)  # +1 porque incluye el mismo
    inds = indices[0].tolist()
    dists = distances[0].tolist()
    # remover el propio listing
    if inds[0] == idx:
        inds = inds[1:]
        dists = dists[1:]
    rows = []
    target_amen = set(a for a in amen_list if df.at[idx, f'amenity__{a}'] == 1)
    for i, dist in zip(inds, dists):
        comp_id = df.at[i, 'id']
        comp_name = df.at[i, 'name']
        comp_amen = set(a for a in amen_list if df.at[i, f'amenity__{a}'] == 1)
        shared = target_amen & comp_amen
        only_target = target_amen - comp_amen
        only_comp = comp_amen - target_amen
        rows.append({
            'comp_index': i,
            'id': comp_id,
            'name': comp_name,
            'distance': float(dist),
            'shared_amen_count': len(shared),
            'only_target_count': len(only_target),
            'only_comp_count': len(only_comp),
            'shared_amenities': ', '.join(sorted(shared)),
            'only_target_amenities': ', '.join(sorted(only_target)),
            'only_comp_amenities': ', '.join(sorted(only_comp))
        })
    return pd.DataFrame(rows).sort_values('distance').reset_index(drop=True)



In [ ]:
# Pipeline: crear features, entrenar (o cargar) modelo
df = listings_df.copy()
df, top_amenities = build_amenity_features(df, top_n=40)
X, num_cols = build_feature_matrix(df, top_amenities)
scaler, knn = train_or_load_knn(X, model_path='models/knn_competitors.pkl', n_neighbors=21)


In [98]:
# Función de uso sencillo
def compara_amenidades_por_id(listing_id, k=10):
    comps = get_competitors(listing_id, df, X, scaler, knn, top_amenities, k=k)
    target_idx = int(df.index[df['id'] == listing_id][0])
    print(f"Listing objetivo: id={listing_id} | name={df.at[target_idx,'name']}\n")
    print("Amenidades objetivo (top seleccionadas):")
    target_amen = [a for a in top_amenities if df.at[target_idx, f'amenity__{a}']==1]
    print(', '.join(target_amen) if target_amen else '<ninguna en top amenidades>')
    print("\nCompetidores encontrados (resumen):")
    display_cols = ['id','name','distance','shared_amen_count','only_target_count','only_comp_count']
    return comps[display_cols].head(k)

In [99]:
# Ejemplo: reemplaza 35797 por el id que quieras analizar
comps_df = compara_amenidades_por_id(1079323346549332893, k=10)
comps_df

Listing objetivo: id=1079323346549332893 | name=Suite independiente en CDMX

Amenidades objetivo (top seleccionadas):
kitchen, wifi, hot water, dishes and silverware, cooking basics, microwave, self check-in, tv, first aid kit, fire extinguisher, exterior security cameras on property, washer

Competidores encontrados (resumen):


,id,name,distance,shared_amen_count,only_target_count,only_comp_count
0,933422028867559779,Loft Amueblado y Equipado,3.82,9,3,5
1,31312417,Cozumel,4.26,7,5,8
2,1417424690359800072,111-Loft a 10 min AICM y a a 10 min centro CDMX,4.47,8,4,3
3,27754625,Habitaciones en excelente departamento,4.52,7,5,5
4,1349606110807067374,Aeropuerto TAPO Estadio GNP,5.32,6,6,3
5,1267477717649709259,Cozumel 32 Habitación 2 de 11m2,5.39,3,9,7
6,856657085496276518,Casa Manuel México Boutique house Gato Room,5.48,5,7,9
7,1051628152679129116,Habitación a 5 calles del WTC Amarilla,5.83,7,5,4
8,25519539,Habitación en Depto. al lado d CU se ve F Médi...,5.84,7,5,6
9,950755074754295083,Hacienda 20,5.90,5,7,6


In [100]:
# converto to json
comps_df.to_json()

'{"id":{"0":933422028867559779,"1":31312417,"2":1417424690359800072,"3":27754625,"4":1349606110807067374,"5":1267477717649709259,"6":856657085496276518,"7":1051628152679129116,"8":25519539,"9":950755074754295083},"name":{"0":"Loft Amueblado y Equipado","1":"Cozumel","2":"111-Loft a 10 min AICM y a a 10 min centro CDMX","3":"Habitaciones en excelente departamento","4":"Aeropuerto TAPO Estadio GNP","5":"Cozumel 32 Habitaci\\u00f3n 2 de 11m2","6":"Casa Manuel M\\u00e9xico Boutique house Gato Room","7":"Habitaci\\u00f3n a 5 calles del WTC Amarilla","8":"Habitaci\\u00f3n en Depto. al lado d CU se ve F M\\u00e9dicina","9":"Hacienda 20"},"distance":{"0":3.8229818211,"1":4.2597147282,"2":4.4737230165,"3":4.5221543792,"4":5.3199718676,"5":5.3857917349,"6":5.4777097188,"7":5.8322861457,"8":5.8403585251,"9":5.8958765669},"shared_amen_count":{"0":9,"1":7,"2":8,"3":7,"4":6,"5":3,"6":5,"7":7,"8":7,"9":5},"only_target_count":{"0":3,"1":5,"2":4,"3":5,"4":6,"5":9,"6":7,"7":5,"8":5,"9":7},"only_comp_cou

# Analiza reviews

In [ ]:
import sqlite3
import pandas as pd
import json 


def parse_reviews_to_dict(text):
    """Convierte 'user:review||user2:review2' en {user: review, ...}"""
    result = {}
    if pd.isna(text) or text.strip() == "":
        return result
    
    for pair in text.split("||"):
        if ":" not in pair:
            continue
        user, comment = pair.split(":", 1)
        result[user.strip()] = comment.strip()
    return result


# load reviews from database
def _get_text_reviews_by_id(listing_id):

    # connect with the database and query the reviews from the listing_id
    conn = sqlite3.connect("db/airbnb.db")
    query = f"""SELECT * FROM reviews WHERE listing_id = {listing_id} AND año_trimestre >= 20200"""
    reviews_df = pd.read_sql_query(query, con=conn)
    conn.close()

    # if there are no comment, return text "No reviews yet"
    if reviews_df.shape == (0, 3):
        return "No reviews yet"
    
    # Replace wrong characters
    reviews_df["all_comments"] = reviews_df["all_comments"].map(lambda x: str(x).replace('<br/>', ''))
    
    # count number reviews
    reviews_df["no_reviews"] = reviews_df["all_comments"].map(lambda x: len(str(x).split('||')))

    # Creamos el diccionario final
    aniomes_json = {}

    for _, row in reviews_df.iterrows():
        aniomes = str(row["año_trimestre"])
        review_dict = parse_reviews_to_dict(row["all_comments"])

        # Si ya existen entradas para ese aniomes, las acumulamos
        if aniomes not in aniomes_json:
            aniomes_json[aniomes] = {}

        # Merge de usuarios/comentarios dentro del aniomes
        aniomes_json[aniomes].update(review_dict)

    # Convertir a JSON (opcional)
    # aniomes_json_str = json.dumps(aniomes_json, ensure_ascii=False, indent=2)
    
    return aniomes_json


aniomes_json_str = _get_text_reviews_by_id(listing_id=44616)

In [11]:
type(aniomes_json_str)

dict

# Structured outputs

Analisis de reviews 

In [1]:
import os
import requests
from dotenv import load_dotenv 
from openai import OpenAI
from typing import Optional, List
from pydantic import BaseModel, Field

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)
model_openai = "gpt-5.1"

In [3]:
class ReviewsCallInsights(BaseModel):
    """
    Output estructurado con insights clave en español  extraidos de reviews de usuarios.
    """
    listing_id: Optional[int] = Field(description="ID del listing de Airbnb, ej. 44616")
    sentiment : Optional[str] = Field(description="Sentimiento general de las reviews (positivo, negativo, neutral)")
    summary: str = Field(description="Resumen de las reviews destacando puntos clave")
    common_themes: List[str] = Field(description="Temas comunes mencionados en las reviews")
    pros: List[str] = Field(description="Aspectos positivos destacados en las reviews")
    cons: List[str] = Field(description="Aspectos negativos destacados en las reviews")
    suggestions: List[str] = Field(description="Sugerencias de mejora basadas en las reviews")
    

In [4]:
def render_transcript(d: dict) -> str:
    """Renderiza el dict de salida en un formato legible."""
    lines = []
    for key, value in d.items():
        if isinstance(value, list):
            lines.append(f"{key.capitalize()}:")
            for item in value:
                lines.append(f" - {item}")
        else:
            lines.append(f"{key.capitalize()}: {value}")
    return "\n".join(lines)

In [5]:
def extract_insights(client, listing_id: int, model_openai) -> ReviewsCallInsights:
    """
    Obtiene insights clave de reviews de usuarios de Airbnb en español usando OpenAI.
    Arguments:
    - client: instancia del cliente OpenAI
    - reviews_text: diccionario con reviews de usuarios
    - model_openai: nombre del modelo OpenAI a usar
    Returns:
    - un objeto ReviewsCallInsights con los insights extraidos.
    """

    reviews_text = _get_text_reviews_by_id(listing_id=listing_id)
    
    transcript_text = render_transcript(reviews_text)

    response = client.chat.completions.parse(
        model=model_openai,
        messages=[
            {"role": "system", "content": "Eres un asistente útil que extrae insights de los reviews de un listing de airbnb. Devuelve solo un JSON valido que siga exactamente el esquema de ReviewsCallInsights. Salidas en español"},
            {"role": "user", "content": transcript_text}
        ],
        response_format=ReviewsCallInsights
    )

    response_text = response.choices[0].message.parsed
    
    # Parsear la respuesta JSON a ReviewsCallInsights
    # insights = ReviewsCallInsights.model_validate_json(response_text)
    
    return response_text.model_dump()

In [8]:
insights = extract_insights(client, 44616, model_openai)

In [23]:
insights

{'listing_id': None,
 'sentiment': 'positivo',
 'summary': 'Las reviews describen a Condesa Haus como una casona histórica con mucho encanto, ideal para grupos grandes y también para parejas o familias, ubicada en una de las mejores zonas de La Condesa. Los huéspedes destacan de forma consistente la excelente ubicación, el diseño y decoración del lugar, la comodidad de las habitaciones con baño privado, y la atención cálida y muy servicial del staff (especialmente Fernando y Josue). Se valora mucho el ambiente tranquilo para volver después de salir, el rooftop, y los desayunos cuando se ofrece el servicio tipo BnB. Hay pocas críticas: algunos detalles menores de mantenimiento, las limitaciones propias de un edificio antiguo (paredes delgadas, control de agua), y un huésped que no regresaría sin detallar el motivo.',
 'common_themes': ['Ubicación excelente en La Condesa, zona bonita y segura',
  'Ideal para grupos grandes (bachelorette, viajes entre amigos, grupos de 9–13 personas)',
  